# AllnAction RoboTaxi — Auto-Prelabel 11 Lớp Bằng SAM 3 (Text-Prompted)

Notebook này dùng **Ultralytics SAM 3 (`SAM3SemanticPredictor`)** để tự động khoanh mask polygon cho **cả 11 lớp giao thông** — đặc biệt cứu các lớp chưa có nhãn như **`3: Other vehicle` (xích lô, xe ba gác, tuk-tuk, rơ-moóc)**.

**Đầu ra:** `sam3_prelabeled.zip` (chuẩn `YOLO Segmentation 1.0`, có sẵn `data.yaml`) sẵn sàng import vào **CVAT** để review hoặc gộp thẳng vào `team780`.

In [ ]:
#@title 1. Cài đặt Ultralytics hỗ trợ SAM 3 & kiểm tra GPU
!pip install -qU "ultralytics>=8.3.237" opencv-python pyyaml huggingface_hub
import torch
print("CUDA available:", torch.cuda.is_available(), "| Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
#@title 2. Kéo repo & tải ảnh (Đèn giao thông VN / Other vehicle)
MODE = "vn_traffic_lights"  #@param ["vn_traffic_lights", "other_vehicle", "both"]
!git clone -q https://github.com/tiennguyeniotd22-ops/allnaction-traffic-seg.git /content/repo || (cd /content/repo && git pull -q)
!mkdir -p /content/raw_images
if MODE in ("vn_traffic_lights", "both"):
    !python /content/repo/scripts/download_vn_traffic_lights.py --out /content/raw_images --per-query 8
if MODE in ("other_vehicle", "both"):
    !python /content/repo/scripts/download_other_vehicle_images.py --out /content/raw_images --per-query 12
!ls -lh /content/raw_images | head -n 20

In [ ]:
#@title 3. Cấu hình SAM 3 weights (sam3.pt)
import os
from getpass import getpass
HF_TOKEN = ""  #@param {type:"string"}
SAM3_WEIGHTS = "sam3.pt"  #@param {type:"string"}
IMAGES_DIR = "/content/raw_images"  #@param {type:"string"}
CONF_THRES = 0.28  #@param {type:"number"}
IMGSZ = 1008  #@param {type:"integer"}

token = HF_TOKEN.strip()
if not token:
    try:
        from google.colab import userdata
        token = (userdata.get("HF_TOKEN") or "").strip()
    except Exception:
        pass
if not token:
    token = getpass("Nhập HF_TOKEN (hoặc Enter nếu đã có sam3.pt): ").strip()

if token:
    os.environ["HF_TOKEN"] = token
    from huggingface_hub import hf_hub_download
    SAM3_WEIGHTS = hf_hub_download(repo_id="facebook/sam3", filename="sam3.pt", token=token)
print("Using SAM3 weights:", SAM3_WEIGHTS)

In [ ]:
#@title 4. Chạy SAM 3 Prelabel 11 lớp & Xuất YOLO-Seg (.zip)
import shutil
from collections import Counter
from pathlib import Path
import cv2
import numpy as np
import yaml
from ultralytics.models.sam import SAM3SemanticPredictor

CLASSES = {
    0: "Car", 1: "Bus", 2: "Truck", 3: "Other vehicle",
    4: "Motorcycle", 5: "Motorcyclist", 6: "Bicycle", 7: "Cyclist",
    8: "Pedestrian", 9: "Sign", 10: "Traffic light",
}

# Nhóm prompt text cho SAM 3 (tách riêng person & 2-wheeler để tự gộp Motorcyclist/Cyclist)
PROMPT_GROUPS = [
    ("car", 0), ("bus", 1), ("truck", 2),
    ("auto rickshaw", 3), ("tricycle", 3), ("cyclo", 3), ("tuk tuk", 3), ("trailer", 3),
    ("motorcycle", 4), ("bicycle", 6), ("person", 8),
    ("traffic sign", 9), ("traffic light", 10),
]

def mask_to_norm_poly(mask_bin: np.ndarray, w: int, h: int, min_area_640: float = 65.0) -> np.ndarray | None:
    contours, _ = cv2.findContours(mask_bin.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    largest = max(contours, key=cv2.contourArea)
    peri = cv2.arcLength(largest, True)
    approx = cv2.approxPolyDP(largest, 0.002 * peri, True).reshape(-1, 2).astype(np.float32)
    if len(approx) < 3:
        return None
    scale = 640.0 / float(max(w, h))
    if cv2.contourArea(approx * scale) < min_area_640:
        return None
    approx[:, 0] = np.clip(approx[:, 0] / float(w), 0.0, 1.0)
    approx[:, 1] = np.clip(approx[:, 1] / float(h), 0.0, 1.0)
    return approx

def masks_overlap(m_person: np.ndarray, m_bike: np.ndarray) -> bool:
    inter = np.logical_and(m_person > 0, m_bike > 0).sum()
    if inter > 0:
        return True
    # Kiểm tra giãn nở nhẹ 8px nếu người ngồi sát yên xe
    dil = cv2.dilate(m_bike.astype(np.uint8), np.ones((9, 9), np.uint8))
    return np.logical_and(m_person > 0, dil > 0).sum() >= 30

predictor = SAM3SemanticPredictor(overrides=dict(
    conf=CONF_THRES, task="segment", mode="predict",
    imgsz=IMGSZ, model=SAM3_WEIGHTS, half=True, save=False, verbose=False
))

out_dir = Path("/content/sam3_prelabeled")
img_out = out_dir / "images" / "train"
lbl_out = out_dir / "labels" / "train"
img_out.mkdir(parents=True, exist_ok=True)
lbl_out.mkdir(parents=True, exist_ok=True)

with open(out_dir / "data.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump({"path": ".", "train": "images/train", "val": "images/train", "nc": 11, "names": CLASSES}, f, sort_keys=False, allow_unicode=True)

img_paths = sorted(p for p in Path(IMAGES_DIR).glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
stats = Counter()
prompts = [p[0] for p in PROMPT_GROUPS]

for idx, img_p in enumerate(img_paths, start=1):
    img = cv2.imread(str(img_p))
    if img is None:
        continue
    h, w = img.shape[:2]
    predictor.set_image(str(img_p))
    results = predictor(text=prompts)

    persons, bikes, others = [], [], []
    if results and results[0].masks is not None:
        res = results[0]
        masks_np = res.masks.data.cpu().numpy()
        cls_idx = res.boxes.cls.cpu().numpy().astype(int) if res.boxes is not None else []
        for m, pid in zip(masks_np, cls_idx):
            m_bin = cv2.resize((m > 0.5).astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST)
            comp_cls = PROMPT_GROUPS[pid][1]
            if comp_cls == 8:
                persons.append(m_bin)
            elif comp_cls in (4, 6):
                bikes.append((comp_cls, m_bin))
            else:
                others.append((comp_cls, m_bin))

    used_p = set()
    final_masks = []
    for b_cls, b_mask in bikes:
        matched = [pi for pi, pm in enumerate(persons) if pi not in used_p and masks_overlap(pm, b_mask)]
        if matched:
            merged = b_mask.copy()
            for pi in matched:
                used_p.add(pi)
                merged = np.logical_or(merged > 0, persons[pi] > 0).astype(np.uint8)
            final_masks.append((5 if b_cls == 4 else 7, merged))
        else:
            final_masks.append((b_cls, b_mask))

    for pi, pm in enumerate(persons):
        if pi not in used_p:
            final_masks.append((8, pm))
    final_masks.extend(others)

    lines = []
    for cid, m_bin in final_masks:
        poly = mask_to_norm_poly(m_bin, w, h)
        if poly is None:
            continue
        lines.append(f"{cid} " + " ".join(f"{x:.6f} {y:.6f}" for x, y in poly))
        stats[CLASSES[cid]] += 1

    shutil.copy2(img_p, img_out / img_p.name)
    (lbl_out / f"{img_p.stem}.txt").write_text("\n".join(lines), encoding="utf-8")
    if idx % 10 == 0 or idx == len(img_paths):
        print(f"Processed {idx}/{len(img_paths)} images...")

zip_path = shutil.make_archive("/content/sam3_prelabeled", "zip", root_dir=out_dir)
print("\n=== SAM 3 PRELABEL SUMMARY ===")
print(dict(stats))
print("Saved:", zip_path)

In [ ]:
#@title 5. Tải file sam3_prelabeled.zip về máy (để import CVAT hoặc gộp vào team780)
from google.colab import files
files.download("/content/sam3_prelabeled.zip")